# SovereignNation — Multi-Agent QLoRA Fine-Tune (Kaggle)

Free GPU (T4 x2) or TPU v3-8. 30 hrs/week quota.

**Before running:**
1. Kaggle → Settings → Secrets → add `HF_TOKEN`
2. Select **GPU T4 x2** in the accelerator dropdown (top right)
3. Set `AGENT` and `MODE` in the Config cell below
4. Run All

| Agent | Role | HF Output |
|-------|------|-----------|
| `avery` | Business strategist (KAIROS) | tastytator/avery-sovereign-lora |
| `forge` | Code generation | tastytator/forge-sovereign-lora |
| `oracle` | Memory & retrieval | tastytator/oracle-sovereign-lora |
| `codex` | Documentation | tastytator/codex-sovereign-lora |
| `sentinel` | Security review | tastytator/sentinel-sovereign-lora |
| `nexus` | Orchestration | tastytator/nexus-sovereign-lora |
| `all` | All agents combined | tastytator/sovereign-agents-lora |

In [ ]:
# ── 0. Hardware check ─────────────────────────────────────────────────────────
import subprocess, os
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
    BACKEND = 'cuda'
else:
    try:
        import torch_xla.core.xla_model as xm
        print('TPU:', xm.xla_device())
        BACKEND = 'tpu'
    except ImportError:
        print('WARNING: No accelerator — CPU only (very slow)')
        BACKEND = 'cpu'
print('Backend:', BACKEND)

In [ ]:
# ── 1. Config — EDIT THESE ────────────────────────────────────────────────────
AGENT  = 'avery'   # avery | forge | oracle | codex | sentinel | nexus | all
MODE   = 'orpo'    # sft | orpo | dpo | grpo
EPOCHS = 3

In [ ]:
# ── 2. Agent definitions ──────────────────────────────────────────────────────
AGENT_CONFIGS = {
    'avery': {
        'system': (
            'You are Avery, the sovereign business strategist for SovereignNation — '
            'a fixed-cost AI platform built for lower and middle class families, '
            'children education, and affordable connectivity. '
            'Use the KAIROS framework: Kickoff, Alignment, Implementation, '
            'Refinement, Optimization, Scaling. Be direct, structured, actionable.'
        ),
        'hf_repo': 'tastytator/avery-sovereign-lora',
        'hf_config': 'dpo',
        'hf_splits': ['bootstrap_dpo', 'spin_business_dpo'],
        'sft_config': 'sft',
        'sft_split': 'train',
    },
    'forge': {
        'system': (
            'You are FORGE, the sovereign code generation specialist for SovereignNation. '
            'Write production-ready Python, JavaScript, and TypeScript. '
            'Always include imports, error handling, type hints, and comments for non-obvious logic.'
        ),
        'hf_repo': 'tastytator/forge-sovereign-lora',
        'hf_config': 'agents',
        'hf_splits': ['train'],
        'agent_filter': 'forge',
    },
    'oracle': {
        'system': (
            'You are ORACLE, the sovereign memory and retrieval specialist for SovereignNation. '
            'Synthesize information into precise structured answers. '
            'Cite source type (memory / document / inference). Be concise.'
        ),
        'hf_repo': 'tastytator/oracle-sovereign-lora',
        'hf_config': 'agents',
        'hf_splits': ['train'],
        'agent_filter': 'oracle',
    },
    'codex': {
        'system': (
            'You are CODEX, the sovereign documentation specialist for SovereignNation. '
            'Write clear complete technical documentation with markdown headings, '
            'code blocks, and working examples. Be accurate and immediately actionable.'
        ),
        'hf_repo': 'tastytator/codex-sovereign-lora',
        'hf_config': 'agents',
        'hf_splits': ['train'],
        'agent_filter': 'codex',
    },
    'sentinel': {
        'system': (
            'You are SENTINEL, the sovereign security specialist for SovereignNation. '
            'Review code and systems for vulnerabilities. Reference OWASP Top 10 and CWE. '
            'State: vulnerability, impact (low/med/high/critical), and the specific fix.'
        ),
        'hf_repo': 'tastytator/sentinel-sovereign-lora',
        'hf_config': 'agents',
        'hf_splits': ['train'],
        'agent_filter': 'sentinel',
    },
    'nexus': {
        'system': (
            'You are NEXUS, the sovereign orchestration specialist for SovereignNation. '
            'Coordinate agents and design workflows. Output structured task graphs: '
            'sequence, parallelism, dependencies, and which agent owns each step.'
        ),
        'hf_repo': 'tastytator/nexus-sovereign-lora',
        'hf_config': 'agents',
        'hf_splits': ['train'],
        'agent_filter': 'nexus',
    },
    'all': {
        'system': 'You are a sovereign AI specialist. Your role is determined by the system context.',
        'hf_repo': 'tastytator/sovereign-agents-lora',
        'hf_config': 'agents',
        'hf_splits': ['train'],
    },
}

cfg    = AGENT_CONFIGS[AGENT]
SYSTEM = cfg['system']
HF_REPO_OUT = cfg['hf_repo']
print(f'Agent  : {AGENT}')
print(f'Mode   : {MODE}')
print(f'Output : {HF_REPO_OUT}')

In [ ]:
# ── 3. Install deps ───────────────────────────────────────────────────────────
if BACKEND == 'cuda':
    os.system('pip install -q "unsloth[colab-new]" trl datasets peft bitsandbytes accelerate transformers')
else:
    os.system('pip install -q trl datasets transformers peft accelerate')
print('Deps installed.')

In [ ]:
# ── 4. Secrets ────────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
HF_TOKEN   = UserSecretsClient().get_secret('HF_TOKEN')
HF_DATASET = 'tastytator/sovereign-economy'
OUTPUT_DIR = '/kaggle/working/sovereign-lora'
MAX_SEQ    = 2048
print('HF token loaded.')

In [ ]:
# ── 5. Load dataset ───────────────────────────────────────────────────────────
from datasets import load_dataset

def try_load(split, config=None):
    kwargs = {'split': split, 'token': HF_TOKEN}
    if config:
        kwargs['name'] = config
    try:
        ds = load_dataset(HF_DATASET, **kwargs)
        print(f'  Loaded {config}/{split}: {len(ds)} rows  cols={ds.column_names}')
        return ds
    except Exception as e:
        print(f'  {config}/{split} not available: {type(e).__name__}')
        return None

ds = None
final_mode = MODE

# Specialist agents: load from agents config and filter
if AGENT not in ('avery', 'all'):
    ds = try_load('train', config='agents')
    if ds is not None:
        agent_filter = cfg.get('agent_filter', AGENT)
        ds = ds.filter(lambda row: row.get('agent', '') == agent_filter)
        print(f'  Filtered to agent={agent_filter}: {len(ds)} rows')
        if len(ds) < 5:
            print(f'  WARNING: Only {len(ds)} rows — run agents_bootstrap.py first!')
            ds = None
elif AGENT == 'all':
    ds = try_load('train', config='agents')

# Avery / fallback: use DPO splits
if ds is None and MODE in ('orpo', 'dpo'):
    for split in cfg.get('hf_splits', ['bootstrap_dpo']):
        ds = try_load(split, config=cfg.get('hf_config', 'dpo'))
        if ds is not None and 'chosen' in ds.column_names:
            break
        ds = None

# SFT fallback
if ds is None:
    print('  DPO splits unavailable — falling back to SFT')
    final_mode = 'sft'
    ds = try_load(cfg.get('sft_split', 'train'), config=cfg.get('sft_config', 'sft'))

assert ds is not None, 'No dataset found! Run pre_train.py on TatorTot first.'
MODE = final_mode
print(f'\nUsing mode={MODE}  agent={AGENT}  rows={len(ds)}')

In [ ]:
# ── 6a. Load model — CUDA (QLoRA via unsloth) ─────────────────────────────────
BASE_MODEL = 'Qwen/Qwen2-7B-Instruct'

if BACKEND == 'cuda':
    from unsloth import FastLanguageModel
    import torch

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ,
        dtype=None, load_in_4bit=True, token=HF_TOKEN,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=32, lora_dropout=0.05, bias='none',
        use_gradient_checkpointing='unsloth',
    )
    model.print_trainable_parameters()

In [ ]:
# ── 6b. Load model — TPU (BF16 LoRA via PEFT) ────────────────────────────────
if BACKEND == 'tpu':
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import get_peft_model, LoraConfig, TaskType
    import torch, torch_xla.core.xla_model as xm

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, token=HF_TOKEN, torch_dtype=torch.bfloat16
    )
    model = get_peft_model(model, LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    ))
    model.print_trainable_parameters()
    model = model.to(xm.xla_device())

In [ ]:
# ── 7. Format + train ─────────────────────────────────────────────────────────
import torch
from transformers import TrainingArguments

def _wrap(goal):
    return (f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
            f'<|im_start|>user\n{goal}<|im_end|>\n'
            f'<|im_start|>assistant\n')

USE_BF16 = (BACKEND != 'cpu')
USE_FP16 = False

if MODE == 'sft':
    from trl import SFTTrainer

    def fmt(row):
        g = str(row.get('instruction') or row.get('goal') or row.get('prompt') or '')
        r = str(row.get('response') or row.get('chosen') or '')
        return {'text': _wrap(g) + r + '<|im_end|>'}

    ds_fmt  = ds.map(fmt, remove_columns=ds.column_names)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=ds_fmt, dataset_text_field='text',
        max_seq_length=MAX_SEQ,
        args=TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            num_train_epochs=EPOCHS, learning_rate=2e-4,
            bf16=USE_BF16, fp16=USE_FP16, logging_steps=10,
            output_dir=OUTPUT_DIR, warmup_ratio=0.1,
            lr_scheduler_type='cosine', save_strategy='epoch', report_to='none',
        ),
    )

elif MODE == 'orpo':
    from trl import ORPOTrainer, ORPOConfig

    def fmt_orpo(row):
        g = str(row.get('goal') or row.get('instruction') or '')
        p = str(row.get('prompt') or f'GOAL: {g}\n\nProvide a detailed sovereign response:')
        return {
            'prompt':   _wrap(p),
            'chosen':   str(row['chosen']) + '<|im_end|>',
            'rejected': str(row['rejected']) + '<|im_end|>',
        }

    ds_fmt  = ds.map(fmt_orpo, remove_columns=ds.column_names)
    trainer = ORPOTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds_fmt,
        args=ORPOConfig(
            max_length=MAX_SEQ, max_prompt_length=512, beta=0.1,
            per_device_train_batch_size=1, gradient_accumulation_steps=8,
            num_train_epochs=EPOCHS, learning_rate=8e-6,
            bf16=USE_BF16, fp16=USE_FP16, logging_steps=10,
            output_dir=OUTPUT_DIR, warmup_ratio=0.1,
            lr_scheduler_type='cosine', save_strategy='epoch', report_to='none',
        ),
    )

elif MODE == 'dpo':
    from trl import DPOTrainer, DPOConfig

    def fmt_dpo(row):
        g = str(row.get('goal') or row.get('instruction') or '')
        p = str(row.get('prompt') or f'GOAL: {g}\n\nProvide a detailed sovereign response:')
        return {
            'prompt':   _wrap(p),
            'chosen':   str(row['chosen']) + '<|im_end|>',
            'rejected': str(row['rejected']) + '<|im_end|>',
        }

    ds_fmt  = ds.map(fmt_dpo, remove_columns=ds.column_names)
    trainer = DPOTrainer(
        model=model, ref_model=None, tokenizer=tokenizer, train_dataset=ds_fmt,
        args=DPOConfig(
            max_length=MAX_SEQ, max_prompt_length=512, beta=0.1,
            per_device_train_batch_size=1, gradient_accumulation_steps=8,
            num_train_epochs=EPOCHS, learning_rate=5e-6,
            bf16=USE_BF16, fp16=USE_FP16, logging_steps=10,
            output_dir=OUTPUT_DIR, warmup_ratio=0.1,
            lr_scheduler_type='cosine', save_strategy='epoch', report_to='none',
        ),
    )

elif MODE == 'grpo':
    from trl import GRPOTrainer, GRPOConfig
    try:
        from unsloth import PatchFastRL
        PatchFastRL('GRPO', FastLanguageModel)
    except (ImportError, AttributeError, NameError):
        pass

    def _reward(completions, **kwargs):
        scores = []
        for text in completions:
            s = 0.0
            if len(text) > 400: s += 1.0
            elif len(text) > 150: s += 0.5
            else: s -= 0.5
            if text.count('\n') > 3: s += 0.5
            scores.append(s)
        return scores

    def fmt_grpo(row):
        p = str(row.get('prompt') or row.get('instruction') or row.get('goal') or '')
        return {'prompt': _wrap(p)}

    ds_fmt  = ds.map(fmt_grpo, remove_columns=ds.column_names)
    ds_fmt  = ds_fmt.filter(lambda r: len(r['prompt']) > 20)
    trainer = GRPOTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds_fmt,
        reward_funcs=[_reward],
        args=GRPOConfig(
            max_prompt_length=512, max_completion_length=512, num_generations=4,
            per_device_train_batch_size=1, gradient_accumulation_steps=8,
            num_train_epochs=EPOCHS, learning_rate=5e-6,
            bf16=USE_BF16, fp16=USE_FP16, logging_steps=10,
            output_dir=OUTPUT_DIR, warmup_ratio=0.1,
            lr_scheduler_type='cosine', save_strategy='epoch', report_to='none',
        ),
    )

print(f'Starting: {MODE.upper()}  agent={AGENT}  {len(ds_fmt)} examples...')
stats = trainer.train()
loss  = stats.metrics.get('train_loss', 0)
rt    = stats.metrics.get('train_runtime', 0)
print(f'\nTraining complete!  Loss={loss:.4f}  Time={rt/60:.1f} min')

In [ ]:
# ── 8. Save + push to HuggingFace ─────────────────────────────────────────────
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved locally to {OUTPUT_DIR}')

model.push_to_hub(HF_REPO_OUT, token=HF_TOKEN, private=False)
tokenizer.push_to_hub(HF_REPO_OUT, token=HF_TOKEN, private=False)

print()
print('=' * 50)
print(f'  DONE  [{MODE.upper()}  {AGENT.upper()}]')
print(f'  Loss   : {loss:.4f}')
print(f'  Time   : {rt/60:.1f} min')
print(f'  LoRA   : https://huggingface.co/{HF_REPO_OUT}')
print('=' * 50)